<a href="https://colab.research.google.com/github/Efrendcl/BoxiCode/blob/buscador-google-drive/explorador_carpeta_drive_bonito.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import os
import io
import re
import csv
import json
import html
import time
import shutil
import base64
import zipfile
import secrets
import threading
import mimetypes
import xml.etree.ElementTree as ET
from string import Template
from dataclasses import dataclass
from datetime import datetime
from http.server import ThreadingHTTPServer, BaseHTTPRequestHandler

from google.colab import drive, output
from IPython.display import display, HTML, JSON
from PIL import Image as PILImage, ImageOps

# ============================================================
# 1. CONFIGURACIÓN
# ============================================================
NOMBRE_CARPETA = 'Acoplamiento'
DIRECTORIO = f'/content/drive/MyDrive/{NOMBRE_CARPETA}'
INCLUIR_SUBCARPETAS = True
MAX_POR_CATEGORIA = 12                  # cuántos archivos se muestran por sección

EXTENSIONES = {
    "Imágenes": ('.jpg', '.jpeg', '.png', '.gif', '.webp'),
    "Música":   ('.mp3', '.wav', '.ogg', '.m4a', '.aac'),
    "Videos":   ('.mp4', '.webm', '.mov', '.avi', '.mkv'),
}
CATEGORIAS = ["Imágenes", "Música", "Videos", "Documentos"]
ALGORITMO_POR_CATEGORIA = {
    "Imágenes":   "Quick Sort",
    "Música":     "Merge Sort",
    "Videos":     "Insertion Sort",
    "Documentos": "Merge Sort",
}
MOTIVO = {
    "Imágenes":   "Muchas fotos con nombres casi únicos: Quick Sort es el más rápido en promedio.",
    "Música":     "Bibliotecas grandes y parcialmente ordenadas: Merge Sort garantiza n·log n y es estable.",
    "Videos":     "Pocos archivos: Insertion Sort tiene menos sobrecarga en listas pequeñas.",
    "Documentos": "Muchos empates (misma extensión o tamaño): Merge Sort es estable y conserva el orden previo.",
}
ICONO = {"Imágenes": "🖼️", "Música": "🎵", "Videos": "🎬", "Documentos": "📄"}
ICONO_EXT = {'.pdf': '📕', '.doc': '📝', '.docx': '📝', '.txt': '📝', '.xls': '📊', '.xlsx': '📊',
             '.csv': '📊', '.ppt': '📽️', '.pptx': '📽️', '.zip': '🗜️', '.rar': '🗜️', '.json': '🧾'}

# Documentos que el lector integrado sabe abrir
TEXTO_EXT = {'.txt', '.md', '.py', '.json', '.csv', '.tsv', '.log', '.xml', '.html', '.css', '.js',
             '.yaml', '.yml', '.ini'}
LEGIBLES = TEXTO_EXT | {'.docx', '.pdf'}
LIMITE_TEXTO = 300_000                 # caracteres máximos que se leen de un archivo de texto
LIMITE_PDF = 30 * 1024 * 1024          # bytes máximos de un PDF que se envían al lector
LIMITE_MUSICA = 40 * 1024 * 1024       # límites del modo de respaldo (base64)
LIMITE_VIDEO = 60 * 1024 * 1024


# ============================================================
# 2. ESCANEO DE LA CARPETA
# ============================================================
@dataclass
class Archivo:
    nombre: str
    ruta: str
    categoria: str
    tamano: int
    fecha: str
    id: int = -1

    @property
    def extension(self):
        return os.path.splitext(self.nombre)[1].lower()

    @property
    def icono(self):
        return ICONO_EXT.get(self.extension, ICONO[self.categoria]) if self.categoria == "Documentos" \
            else ICONO[self.categoria]


def categoria_de(nombre):
    for categoria, extensiones in EXTENSIONES.items():
        if nombre.lower().endswith(extensiones):
            return categoria
    return "Documentos"


def escanear(directorio):
    lista = []
    for carpeta, subcarpetas, nombres in os.walk(directorio):
        for nombre in nombres:
            ruta = os.path.join(carpeta, nombre)
            try:
                st = os.stat(ruta)
            except OSError:
                continue
            lista.append(Archivo(
                nombre, ruta, categoria_de(nombre), st.st_size,
                datetime.fromtimestamp(st.st_mtime).strftime('%Y-%m-%d %H:%M')))
        if not INCLUIR_SUBCARPETAS:
            break
    for i, a in enumerate(lista):
        a.id = i
    return lista


def formatear_tamano(n):
    for unidad in ("B", "KB", "MB"):
        if n < 1024:
            return f"{n:.0f} {unidad}" if unidad == "B" else f"{n:.1f} {unidad}"
        n /= 1024
    return f"{n:.1f} GB"


# ============================================================
# 3. ALGORITMOS DE ORDENAMIENTO
# ============================================================
comparaciones = [0]


def quick_sort(lista, clave):
    if len(lista) <= 1:
        return lista
    pivote = clave(lista[len(lista) // 2])
    menores, iguales, mayores = [], [], []
    for x in lista:
        comparaciones[0] += 1
        (menores if clave(x) < pivote else iguales if clave(x) == pivote else mayores).append(x)
    return quick_sort(menores, clave) + iguales + quick_sort(mayores, clave)


def merge_sort(lista, clave):
    if len(lista) <= 1:
        return lista
    medio = len(lista) // 2
    izq, der = merge_sort(lista[:medio], clave), merge_sort(lista[medio:], clave)
    resultado, i, j = [], 0, 0
    while i < len(izq) and j < len(der):
        comparaciones[0] += 1
        if clave(izq[i]) <= clave(der[j]):
            resultado.append(izq[i]); i += 1
        else:
            resultado.append(der[j]); j += 1
    return resultado + izq[i:] + der[j:]


def bubble_sort(lista, clave):
    a = lista[:]
    for i in range(len(a) - 1):
        hubo_cambio = False
        for j in range(len(a) - 1 - i):
            comparaciones[0] += 1
            if clave(a[j]) > clave(a[j + 1]):
                a[j], a[j + 1] = a[j + 1], a[j]
                hubo_cambio = True
        if not hubo_cambio:
            break
    return a


def insertion_sort(lista, clave):
    a = lista[:]
    for i in range(1, len(a)):
        actual, j = a[i], i - 1
        while j >= 0:
            comparaciones[0] += 1
            if clave(a[j]) > clave(actual):
                a[j + 1] = a[j]; j -= 1
            else:
                break
        a[j + 1] = actual
    return a


ALGORITMOS = {"Quick Sort": quick_sort, "Merge Sort": merge_sort,
              "Bubble Sort": bubble_sort, "Insertion Sort": insertion_sort}
CAMPOS = {
    "Nombre":      lambda a: a.nombre.lower(),
    "Tamaño":      lambda a: a.tamano,
    "Fecha":       lambda a: a.fecha,
    "Extensión":   lambda a: a.extension,
}


def ordenar(lista, algoritmo, campo, descendente):
    comparaciones[0] = 0
    inicio = time.perf_counter()
    resultado = ALGORITMOS[algoritmo](lista, CAMPOS[campo])
    ms = (time.perf_counter() - inicio) * 1000
    return (resultado[::-1] if descendente else resultado), ms, comparaciones[0]


def ordenar_por_categoria(lista, campo, descendente):
    resultado = {}
    for categoria in CATEGORIAS:
        grupo = [a for a in lista if a.categoria == categoria]
        if grupo:
            algoritmo = ALGORITMO_POR_CATEGORIA[categoria]
            ordenados, ms, comps = ordenar(grupo, algoritmo, campo, descendente)
            resultado[categoria] = (ordenados, algoritmo, ms, comps)
    return resultado


# ============================================================
# 4. DISEÑO EN XML
# ============================================================
# Si existe este archivo en /content se usa; si no, se usa INTERFAZ_XML.
# (Es un nombre nuevo a propósito: así no se carga un XML viejo que no tenga las novedades.)
XML_ARCHIVO = 'interfaz_oceano_v2.xml'

INTERFAZ_XML = r'''<?xml version="1.0" encoding="UTF-8"?>
<!--
  INTERFAZ "OCEANO" v2 (diseno en XML)
    estilos      CSS
    plantillas   HTML con marcas $variable que Python rellena
    diseno       arbol de la pantalla (cada elemento con id lo usa el JavaScript)
    script       comportamiento del navegador
-->
<interfaz>

  <estilos><![CDATA[
#xd-app{--a1:#06b6d4;--a2:#3b82f6;--a3:#8b5cf6;position:relative;width:100%;height:820px;overflow:hidden;
  border-radius:28px;color:inherit;font-family:"Segoe UI",system-ui,-apple-system,Roboto,sans-serif;line-height:1.35}
#xd-app *{box-sizing:border-box}
#app{position:relative;height:100%}
.xd-oculto{display:none!important}
.xd-main{height:100%;overflow-y:auto;overflow-x:hidden;padding:4px 4px 130px;scroll-behavior:smooth;
  scrollbar-width:thin;scrollbar-color:rgba(128,128,128,.45) transparent}
#xd-app:not(.listo) .xd-main{opacity:0}
#xd-app.listo .xd-main{animation:xd-entra .7s cubic-bezier(.2,.8,.2,1) both}

@keyframes xd-entra{from{opacity:0;transform:translateY(24px)}to{opacity:1;transform:none}}
@keyframes xd-up{from{opacity:0;transform:translateY(14px)}to{opacity:1;transform:none}}
@keyframes xd-pop{0%{opacity:0;transform:scale(.4) rotate(-12deg)}70%{transform:scale(1.08) rotate(3deg)}100%{opacity:1;transform:none}}
@keyframes xd-carga{to{width:100%}}
@keyframes xd-ola{to{transform:translateX(-50%)}}
@keyframes xd-shimmer{to{background-position:-200% 0}}
@keyframes xd-gira{to{transform:rotate(360deg)}}
@keyframes xd-eq{0%,100%{transform:scaleY(.25)}50%{transform:scaleY(1)}}

/* ---------- Animación de entrada ---------- */
.xd-splash{position:absolute;inset:0;z-index:60;display:flex;flex-direction:column;align-items:center;justify-content:center;gap:16px;
  color:#fff;background:linear-gradient(135deg,#0891b2,#3b82f6 55%,#7c3aed);
  transition:opacity .7s ease,transform .7s ease,visibility .7s}
.xd-splash.fuera{opacity:0;transform:scale(1.08);visibility:hidden;pointer-events:none}
.xd-splash-logo{position:relative;z-index:1;width:100px;height:100px;border-radius:32px;background:rgba(255,255,255,.22);
  display:flex;align-items:center;justify-content:center;font-size:52px;animation:xd-pop .9s cubic-bezier(.2,1.2,.4,1) both}
.xd-splash h2{position:relative;z-index:1;margin:0;font-size:26px;font-weight:700;animation:xd-up .7s .3s both}
.xd-splash p{position:relative;z-index:1;margin:0;opacity:.85;font-size:14px;animation:xd-up .7s .45s both}
.xd-splash-bar{position:relative;z-index:1;width:230px;height:6px;border-radius:99px;background:rgba(255,255,255,.28);overflow:hidden;animation:xd-up .7s .55s both}
.xd-splash-bar i{display:block;height:100%;width:0;border-radius:99px;background:#fff;animation:xd-carga 1.4s .5s ease-out forwards}
.xd-olas{position:absolute;left:0;bottom:0;width:200%;height:110px;animation:xd-ola 8s linear infinite}
.xd-olas.b{height:80px;animation-duration:5s}

/* ---------- Cabecera ---------- */
.xd-hero{position:relative;overflow:hidden;border-radius:28px;padding:28px 30px;color:#fff;
  background:linear-gradient(120deg,var(--a1),var(--a2) 55%,var(--a3));box-shadow:0 14px 34px rgba(59,130,246,.38)}
.xd-hero::after{content:"";position:absolute;right:-70px;top:-80px;width:240px;height:240px;border-radius:50%;background:rgba(255,255,255,.13)}
.xd-hero::before{content:"";position:absolute;right:120px;bottom:-90px;width:170px;height:170px;border-radius:50%;background:rgba(255,255,255,.09)}
.xd-hero-t{display:flex;align-items:center;gap:16px;position:relative;z-index:1}
.xd-logo{width:62px;height:62px;border-radius:20px;background:rgba(255,255,255,.22);display:flex;align-items:center;justify-content:center;font-size:32px}
.xd-hero h1{margin:0;font-size:28px;font-weight:700;color:#fff}
.xd-hero p{margin:2px 0 0;opacity:.88;font-size:14px}
.xd-stats{display:flex;flex-wrap:wrap;gap:12px;margin-top:20px;position:relative;z-index:1}
.xd-stat{display:flex;align-items:center;gap:8px;background:rgba(255,255,255,.2);border-radius:999px;padding:8px 18px 8px 12px;backdrop-filter:blur(4px)}
#xd-app.listo .xd-stat{animation:xd-pop .6s both}
#xd-app.listo .xd-stat:nth-child(1){animation-delay:.25s}
#xd-app.listo .xd-stat:nth-child(2){animation-delay:.33s}
#xd-app.listo .xd-stat:nth-child(3){animation-delay:.41s}
#xd-app.listo .xd-stat:nth-child(4){animation-delay:.49s}
#xd-app.listo .xd-stat:nth-child(5){animation-delay:.57s}
.xd-stat-i{font-size:18px}
.xd-stat b{font-size:17px}
.xd-stat span:last-child{font-size:13px;opacity:.9}

/* ---------- Pestañas ---------- */
.xd-tabs{display:flex;gap:6px;width:max-content;max-width:100%;margin:18px 0 0;padding:5px;border-radius:999px;
  border:1px solid rgba(128,128,128,.25);background:rgba(128,128,128,.07)}
.xd-tab{padding:9px 20px;border-radius:999px;border:none;background:transparent;color:inherit;cursor:pointer;
  font-size:14px;font-weight:600;font-family:inherit;transition:background .25s,color .25s,box-shadow .25s}
.xd-tab:hover{background:rgba(128,128,128,.16)}
.xd-tab.activo{color:#fff;background:linear-gradient(120deg,var(--a1),var(--a3));box-shadow:0 6px 16px rgba(99,102,241,.4)}
.xd-vista{animation:xd-up .4s both}

/* ---------- Panel de controles ---------- */
.xd-panel{display:flex;flex-direction:column;gap:16px;margin-top:18px;padding:20px;border-radius:26px;
  border:1px solid rgba(128,128,128,.25);background:rgba(128,128,128,.07)}
.xd-input{display:flex;align-items:center;gap:12px;height:54px;padding:0 22px;border-radius:999px;
  border:2px solid rgba(128,128,128,.3);transition:border-color .15s,box-shadow .15s}
.xd-input:focus-within{border-color:var(--a2);box-shadow:0 0 0 5px rgba(59,130,246,.2)}
.xd-input-i{font-size:18px;opacity:.7}
.xd-input input{flex:1;min-width:0;border:none;outline:none;background:transparent;color:inherit;font-size:16px;font-family:inherit}
.xd-input input::placeholder{color:inherit;opacity:.5}

.xd-botones{display:flex;flex-wrap:wrap;gap:10px}
.xd-btn{display:inline-flex;align-items:center;gap:8px;padding:10px 20px;border-radius:999px;cursor:pointer;
  border:2px solid rgba(128,128,128,.32);background:transparent;color:inherit;font-size:15px;font-weight:600;font-family:inherit;
  transition:transform .15s,box-shadow .15s,background .15s}
.xd-btn:hover{transform:translateY(-2px);box-shadow:0 8px 16px rgba(0,0,0,.16);background:rgba(128,128,128,.12)}
.xd-btn:active{transform:translateY(0) scale(.97)}
.xd-btn.activo{color:#fff;border-color:transparent;background:var(--c,linear-gradient(120deg,var(--a1),var(--a3)));box-shadow:0 8px 20px rgba(99,102,241,.38)}
.xd-n{border-radius:999px;padding:1px 10px;font-size:12px;background:rgba(128,128,128,.25)}
.xd-btn.activo .xd-n{background:rgba(255,255,255,.3)}

.xd-controles{display:flex;flex-wrap:wrap;align-items:center;gap:16px 22px}
.xd-select{display:inline-flex;align-items:center;gap:10px;font-size:14px;font-weight:600}
.xd-select select{height:42px;padding:0 38px 0 18px;border-radius:999px;border:2px solid rgba(128,128,128,.32);cursor:pointer;
  background:transparent;color:inherit;font-size:14px;font-family:inherit;appearance:none;-webkit-appearance:none;
  background-image:linear-gradient(45deg,transparent 50%,currentColor 50%),linear-gradient(135deg,currentColor 50%,transparent 50%);
  background-position:calc(100% - 20px) 18px,calc(100% - 14px) 18px;background-size:6px 6px,6px 6px;background-repeat:no-repeat}
.xd-select select:focus{outline:none;border-color:var(--a2)}
.xd-select option{color:#111}
.xd-switch{display:inline-flex;align-items:center;gap:10px;cursor:pointer;font-size:14px;font-weight:600;user-select:none}
.xd-switch input{display:none}
.xd-slider{position:relative;width:46px;height:26px;border-radius:999px;background:rgba(128,128,128,.4);transition:background .2s}
.xd-slider::after{content:"";position:absolute;top:3px;left:3px;width:20px;height:20px;border-radius:50%;background:#fff;
  box-shadow:0 2px 5px rgba(0,0,0,.3);transition:transform .2s}
.xd-switch input:checked + .xd-slider{background:linear-gradient(120deg,var(--a1),var(--a3))}
.xd-switch input:checked + .xd-slider::after{transform:translateX(20px)}

.xd-leg{display:flex;flex-wrap:wrap;align-items:center;gap:8px;font-size:13px}
.xd-algo{display:inline-block;padding:4px 14px;border-radius:999px;font-size:12px;font-weight:600;cursor:help;
  border:1.5px solid var(--c);background:rgba(128,128,128,.08)}
.xd-info{display:inline-block;margin:16px 0 4px;padding:7px 18px;border-radius:999px;font-size:13px;font-weight:600;
  background:linear-gradient(120deg,rgba(6,182,212,.18),rgba(139,92,246,.18))}

/* ---------- Secciones ---------- */
.xd-sec{display:flex;flex-wrap:wrap;align-items:center;gap:10px;margin:26px 0 12px}
.xd-dot{width:14px;height:14px;border-radius:50%;background:var(--c);box-shadow:0 0 0 5px color-mix(in srgb,var(--c) 25%,transparent)}
.xd-sec-t{font-size:19px;font-weight:700}
.xd-badge{padding:2px 12px;border-radius:999px;font-size:12px;font-weight:700;color:#fff;background:var(--c)}
#resultados.xd-anim .xd-card,#resultados.xd-anim .xd-fila,#resultados.xd-anim .xd-sec{animation:xd-up .4s both}

/* Tarjetas de imágenes */
.xd-grid{display:grid;grid-template-columns:repeat(auto-fill,minmax(180px,1fr));gap:18px}
.xd-card{position:relative;overflow:hidden;border-radius:22px;border:1px solid rgba(128,128,128,.25);cursor:zoom-in;
  background:rgba(128,128,128,.08);transition:transform .2s,box-shadow .2s}
.xd-card:hover{transform:translateY(-6px) scale(1.01);box-shadow:0 16px 32px rgba(0,0,0,.3)}
.xd-card img{display:block;width:100%;height:170px;object-fit:cover}
.xd-thumb{height:170px;background:linear-gradient(100deg,rgba(128,128,128,.12) 30%,rgba(128,128,128,.26) 50%,rgba(128,128,128,.12) 70%) 0 0/200% 100%;
  animation:xd-shimmer 1.2s linear infinite}
.xd-thumb.listo{animation:none}
.xd-thumb img{opacity:0;transition:opacity .45s,transform .45s}
.xd-thumb img.ok{opacity:1}
.xd-card:hover .xd-thumb img{transform:scale(1.06)}
.xd-ph{height:170px;display:flex;align-items:center;justify-content:center;font-size:48px}
.xd-cap{padding:11px 14px;font-size:12px}
.xd-cap b{display:block;font-size:13px;white-space:nowrap;overflow:hidden;text-overflow:ellipsis}
.xd-cap span{opacity:.65}

/* Filas */
.xd-fila{display:flex;align-items:center;gap:12px;margin:8px 0}
.xd-row{flex:1;min-width:0;display:flex;align-items:center;gap:14px;padding:11px 18px;border-radius:22px;
  border:1px solid rgba(128,128,128,.25);background:rgba(128,128,128,.08);transition:transform .15s,background .15s,border-color .2s}
.xd-row:hover{background:rgba(128,128,128,.16);transform:translateX(4px)}
.xd-fila.actual .xd-row{border-color:var(--a1);background:rgba(6,182,212,.14)}
.xd-ico{flex:none;width:46px;height:46px;border-radius:16px;display:flex;align-items:center;justify-content:center;font-size:22px}
.xd-txt{min-width:0}
.xd-name{font-weight:600;font-size:14px;white-space:nowrap;overflow:hidden;text-overflow:ellipsis}
.xd-meta{font-size:12px;opacity:.65}
.xd-ext{margin-left:auto;flex:none;padding:4px 12px;border-radius:999px;font-size:11px;font-weight:700;letter-spacing:.6px;
  text-transform:uppercase;background:rgba(128,128,128,.22)}
.xd-play{flex:none;width:54px;height:54px;border-radius:50%;border:none;cursor:pointer;color:#fff;font-size:20px;
  background:linear-gradient(135deg,#22c55e,#14b8a6);box-shadow:0 8px 18px rgba(20,184,166,.45);transition:transform .15s,box-shadow .15s}
.xd-play.xd-leer{background:linear-gradient(135deg,#3b82f6,#8b5cf6);box-shadow:0 8px 18px rgba(99,102,241,.45)}
.xd-play:hover{transform:scale(1.12)}
.xd-play:active{transform:scale(.95)}

.xd-empty{padding:48px 20px;text-align:center;font-size:15px;line-height:1.8;border-radius:26px;
  border:2px dashed rgba(128,128,128,.4);opacity:.85}

/* ---------- Mini reproductor ---------- */
.xd-mini{position:absolute;left:14px;right:14px;bottom:14px;z-index:30;display:flex;flex-direction:column;gap:8px;
  padding:12px 18px;border-radius:28px;color:#fff;background:rgba(15,23,42,.9);backdrop-filter:blur(16px);
  box-shadow:0 20px 44px rgba(0,0,0,.5);border:1px solid rgba(255,255,255,.1);
  transform:translateY(160%);transition:transform .5s cubic-bezier(.2,.9,.3,1.08)}
.xd-mini.activo{transform:none}
.xd-mini-fila{display:flex;align-items:center;gap:12px;min-width:0}
.xd-disco{flex:none;width:46px;height:46px;border-radius:50%;display:flex;align-items:center;justify-content:center;font-size:22px;
  background:conic-gradient(from 0deg,#06b6d4,#8b5cf6,#ec4899,#06b6d4)}
.xd-mini.sonando .xd-disco{animation:xd-gira 4s linear infinite}
.xd-mini-info{flex:1;min-width:0}
.xd-mini-titulo{font-weight:700;font-size:14px;white-space:nowrap;overflow:hidden;text-overflow:ellipsis}
.xd-mini-sub{font-size:12px;opacity:.65}
.xd-eq{flex:none;display:flex;align-items:flex-end;gap:3px;height:22px}
.xd-eq i{width:4px;height:100%;border-radius:2px;background:#22d3ee;transform:scaleY(.25);transform-origin:bottom}
.xd-mini.sonando .xd-eq i{animation:xd-eq .9s ease-in-out infinite}
.xd-mini.sonando .xd-eq i:nth-child(2){animation-delay:.2s}
.xd-mini.sonando .xd-eq i:nth-child(3){animation-delay:.4s}
.xd-mini.sonando .xd-eq i:nth-child(4){animation-delay:.1s}
.xd-mini.es-video .xd-eq{display:none}
.xd-mini:not(.es-video) .xd-solo-video{display:none}
.xd-mini-ctl{display:flex;align-items:center;gap:6px}
.xd-mb{width:38px;height:38px;border-radius:50%;border:none;cursor:pointer;color:#fff;font-size:16px;background:rgba(255,255,255,.12);
  transition:background .15s,transform .15s}
.xd-mb:hover{background:rgba(255,255,255,.26);transform:scale(1.08)}
.xd-mb-play{width:46px;height:46px;font-size:18px;background:linear-gradient(135deg,#22c55e,#14b8a6)}
.xd-vol{display:flex;align-items:center;gap:6px;font-size:14px}
.xd-vol input{width:80px}
.xd-mini-fila span{flex:none;font-size:11px;opacity:.7;min-width:34px;text-align:center}
.xd-rng{-webkit-appearance:none;appearance:none;flex:1;height:6px;border-radius:99px;outline:none;cursor:pointer;
  background:linear-gradient(90deg,#22d3ee var(--p,0%),rgba(255,255,255,.25) var(--p,0%))}
.xd-rng::-webkit-slider-thumb{-webkit-appearance:none;width:14px;height:14px;border-radius:50%;background:#fff;
  box-shadow:0 2px 6px rgba(0,0,0,.45);transition:transform .15s}
.xd-rng:hover::-webkit-slider-thumb{transform:scale(1.35)}
.xd-rng::-moz-range-thumb{width:14px;height:14px;border:none;border-radius:50%;background:#fff}

/* Ventana flotante de video */
.xd-vwin{position:absolute;right:18px;bottom:150px;width:320px;z-index:29;overflow:hidden;border-radius:22px;background:#000;
  box-shadow:0 20px 44px rgba(0,0,0,.5);opacity:0;visibility:hidden;transform:translateY(24px) scale(.92);
  transition:opacity .35s,transform .45s cubic-bezier(.2,.9,.3,1.08),visibility .35s,width .4s,height .4s,left .4s,top .4s}
.xd-vwin.activo{opacity:1;visibility:visible;transform:none}
.xd-vwin video{display:block;width:100%;height:100%;max-height:100%;background:#000;cursor:pointer}
.xd-vwin.grande{left:18px;right:18px;top:18px;bottom:150px;width:auto}
.xd-vwin.grande video{object-fit:contain}

/* ---------- Visor de imágenes ---------- */
.xd-visor{position:absolute;inset:0;z-index:40;display:flex;flex-direction:column;background:rgba(2,6,23,.94);color:#fff;
  backdrop-filter:blur(10px);opacity:0;visibility:hidden;transition:opacity .3s,visibility .3s}
.xd-visor.abierto{opacity:1;visibility:visible}
.xd-visor-top{flex:none;display:flex;align-items:center;gap:10px;padding:14px 20px}
.xd-visor-tit{flex:1;min-width:0}
.xd-visor-tit b{display:block;font-size:15px;white-space:nowrap;overflow:hidden;text-overflow:ellipsis}
.xd-visor-tit span{font-size:12px;opacity:.65}
.xd-visor-cont{flex:none;font-size:13px;opacity:.75;padding:0 6px}
.xd-vb{flex:none;min-width:40px;height:40px;padding:0 12px;border-radius:999px;border:none;cursor:pointer;color:inherit;
  font-size:16px;font-weight:600;background:rgba(255,255,255,.12);transition:background .15s,transform .15s}
.xd-vb:hover{background:rgba(255,255,255,.26);transform:scale(1.06)}
.xd-escena{position:relative;flex:1;min-height:0;display:flex;align-items:center;justify-content:center;overflow:hidden;touch-action:none}
.xd-escena img{max-width:94%;max-height:94%;user-select:none;-webkit-user-drag:none;cursor:grab;opacity:0;
  border-radius:12px;box-shadow:0 20px 60px rgba(0,0,0,.6);transition:transform .25s ease,opacity .35s}
.xd-escena img.ok{opacity:1;animation:xd-pop .45s cubic-bezier(.2,.9,.3,1.1) both}
.xd-escena img.arrastrando{transition:none;cursor:grabbing}
.xd-spin{position:absolute;width:42px;height:42px;border-radius:50%;border:4px solid rgba(255,255,255,.25);border-top-color:#22d3ee;
  animation:xd-gira .8s linear infinite;opacity:0;pointer-events:none}
.xd-escena.cargando .xd-spin{opacity:1}
.xd-nav{position:absolute;top:50%;z-index:2;width:52px;height:52px;margin-top:-26px;border-radius:50%;border:none;cursor:pointer;
  color:#fff;font-size:30px;line-height:1;background:rgba(255,255,255,.14);transition:background .15s,transform .15s}
.xd-nav:hover{background:rgba(255,255,255,.3);transform:scale(1.1)}
.xd-nav-i{left:16px}
.xd-nav-d{right:16px}

/* ---------- Lector de documentos ---------- */
.xd-lector{position:absolute;inset:0;z-index:40;background:rgba(2,6,23,.6);backdrop-filter:blur(8px);
  opacity:0;visibility:hidden;transition:opacity .3s,visibility .3s}
.xd-lector.abierto{opacity:1;visibility:visible}
.xd-lector-hoja{position:absolute;left:3%;right:3%;top:20px;bottom:20px;display:flex;flex-direction:column;overflow:hidden;
  border-radius:26px;background:#fdfdfc;color:#1e293b;box-shadow:0 24px 60px rgba(0,0,0,.5);
  transform:translateY(40px) scale(.97);transition:transform .45s cubic-bezier(.2,.9,.3,1.08)}
.xd-lector.abierto .xd-lector-hoja{transform:none}
.xd-lector-top{flex:none;display:flex;align-items:center;gap:12px;padding:14px 20px;border-bottom:1px solid rgba(100,116,139,.2);background:#f1f5f9}
.xd-lector-top .xd-txt{flex:1}
.xd-lector-top .xd-ico{background:rgba(59,130,246,.15)}
.xd-lector-top .xd-vb{background:rgba(100,116,139,.16);color:#1e293b}
.xd-lector-top .xd-vb:hover{background:rgba(100,116,139,.3)}
.xd-lector-cuerpo{--fs:16px;flex:1;min-height:0;overflow:auto;padding:26px clamp(18px,6%,64px);font-size:var(--fs);line-height:1.7}
.xd-lector-cuerpo p{margin:0 0 .9em;max-width:76ch}
.xd-lector-cuerpo h3{margin:1.3em 0 .5em;font-size:1.25em}
.xd-lector-cuerpo pre{margin:0;white-space:pre-wrap;word-break:break-word;font:inherit;font-family:ui-monospace,Menlo,Consolas,monospace;font-size:.88em;line-height:1.6}
.xd-lector-cuerpo table{border-collapse:collapse;font-size:.88em}
.xd-lector-cuerpo th,.xd-lector-cuerpo td{padding:7px 14px;border:1px solid rgba(100,116,139,.3);text-align:left;white-space:nowrap}
.xd-lector-cuerpo th{background:#e2e8f0;position:sticky;top:-26px}
.xd-lector-cuerpo.pdf{background:#cbd5e1;display:flex;flex-direction:column;align-items:center;gap:14px;padding:18px}
.xd-pagina{background:#fff;box-shadow:0 6px 18px rgba(0,0,0,.25);border-radius:4px;animation:xd-up .35s both}
.xd-nota{padding:10px 16px;border-radius:14px;background:rgba(59,130,246,.12);font-size:.85em}
.xd-cargando{padding:60px 20px;text-align:center;opacity:.7}

/* ---------- Almacenamiento ---------- */
.xd-alm{display:grid;grid-template-columns:repeat(auto-fit,minmax(300px,1fr));gap:18px;margin-top:18px}
.xd-caja{padding:22px;border-radius:26px;border:1px solid rgba(128,128,128,.25);background:rgba(128,128,128,.07)}
.xd-caja h3{margin:0 0 16px;font-size:17px}
.xd-ancha{grid-column:1/-1}
.xd-dona{display:flex;flex-wrap:wrap;align-items:center;justify-content:center;gap:24px}
.xd-dona-svg{position:relative;width:210px;height:210px;flex:none}
.xd-dona-svg svg{width:100%;height:100%;transform:rotate(-90deg)}
.xd-seg{transition:stroke-dasharray 1.1s cubic-bezier(.2,.8,.2,1)}
.xd-dona-c{position:absolute;inset:0;display:flex;flex-direction:column;align-items:center;justify-content:center;text-align:center}
.xd-dona-c b{font-size:24px}
.xd-dona-c span{font-size:12px;opacity:.7}
.xd-leyenda{flex:1;min-width:190px;display:flex;flex-direction:column;gap:10px}
.xd-ley{display:flex;align-items:center;gap:10px;font-size:14px}
.xd-ley i{flex:none;width:12px;height:12px;border-radius:50%}
.xd-ley span{margin-left:auto;opacity:.75;font-size:13px;white-space:nowrap}
.xd-top{display:flex;align-items:center;gap:12px;margin:10px 0;font-size:13px}
.xd-top .n{width:36%;min-width:0;white-space:nowrap;overflow:hidden;text-overflow:ellipsis;font-weight:600}
.xd-barra{flex:1;height:12px;border-radius:99px;background:rgba(128,128,128,.2);overflow:hidden}
.xd-barra i{display:block;height:100%;width:0;border-radius:99px;transition:width 1s cubic-bezier(.2,.8,.2,1)}
.xd-top .t{flex:none;width:68px;text-align:right;opacity:.75}
.xd-drive-barra{height:20px;margin:14px 0 8px}
.xd-drive-barra i{background:linear-gradient(90deg,var(--a1),var(--a3))}

@media (prefers-reduced-motion:reduce){
  #xd-app *,#xd-app *::before,#xd-app *::after{animation-duration:.01ms!important;animation-iteration-count:1!important;transition-duration:.01ms!important}
}
@media (max-width:640px){
  .xd-vol{display:none}
  .xd-vwin{width:220px}
}
  ]]></estilos>

  <plantillas>
    <plantilla id="estadistica"><![CDATA[<div class="xd-stat"><span class="xd-stat-i">$icono</span><b>$n</b><span>$nombre</span></div>]]></plantilla>

    <plantilla id="seccion"><![CDATA[
<div class="xd-sec" style="--c:$color"><span class="xd-dot"></span><span class="xd-sec-t">$icono $categoria</span>
<span class="xd-badge">$total</span><span class="xd-algo" title="$motivo">⚙️ $algoritmo · $ms ms · $comparaciones comparaciones</span></div>
    ]]></plantilla>

    <plantilla id="cuadricula"><![CDATA[<div class="xd-grid">$tarjetas</div>]]></plantilla>

    <plantilla id="tarjeta-imagen"><![CDATA[
<div class="xd-card" data-id="$id"><div class="xd-thumb"><img data-mid="$id" alt=""></div><div class="xd-cap"><b title="$nombre">$nombre</b><span>$tamano · $fecha</span></div></div>
    ]]></plantilla>

    <plantilla id="fila-archivo"><![CDATA[
<div class="xd-fila">$boton<div class="xd-row"><div class="xd-ico" style="background:${color}26">$icono</div>
<div class="xd-txt"><div class="xd-name" title="$nombre">$nombre</div><div class="xd-meta">$tamano · $fecha</div></div>
<span class="xd-ext">$ext</span></div></div>
    ]]></plantilla>

    <plantilla id="boton-play"><![CDATA[<button class="xd-play" data-id="$id" data-cat="$cat" title="Reproducir">▶</button>]]></plantilla>

    <plantilla id="boton-leer"><![CDATA[<button class="xd-play xd-leer" data-id="$id" title="Leer documento">📖</button>]]></plantilla>

    <plantilla id="mas"><![CDATA[<div class="xd-meta" style="margin:6px 4px">… y $cantidad más. Activa «Mostrar todos» o usa el buscador.</div>]]></plantilla>

    <plantilla id="vacio"><![CDATA[<div class="xd-empty">🔎<br><b>Sin resultados</b><br>Prueba con otro nombre o cambia el filtro.</div>]]></plantilla>

    <plantilla id="info"><![CDATA[<span class="xd-info">⏱️ $ms ms en total · $total archivo(s)</span>]]></plantilla>

    <plantilla id="aviso"><![CDATA[<div class="xd-empty">⚠️<br>$mensaje</div>]]></plantilla>

    <plantilla id="leyenda"><![CDATA[<div class="xd-leg">⚙️ Algoritmo automático por tipo: $items</div>]]></plantilla>

    <plantilla id="leyenda-item"><![CDATA[<span class="xd-algo" style="--c:$color" title="$motivo">$icono $categoria → <b>$algoritmo</b></span>]]></plantilla>
  </plantillas>

  <diseno>
    <vertical id="app">

      <vertical id="principal" clase="xd-main">

        <html id="cabecera"><![CDATA[
<div class="xd-hero"><div class="xd-hero-t"><div class="xd-logo">📁</div>
<div><h1>$carpeta</h1><p>Explorador de archivos de Google Drive</p></div></div>
<div class="xd-stats">$estadisticas</div></div>
        ]]></html>

        <pestanas id="vistas">
          <opcion valor="archivos" icono="🗂️" texto="Archivos"/>
          <opcion valor="almacenamiento" icono="📊" texto="Almacenamiento"/>
        </pestanas>

        <vertical id="vista_archivos" clase="xd-vista">
          <vertical id="panel" clase="xd-panel">
            <texto id="buscador" icono="🔍" ayuda="Buscar por nombre..."/>

            <botones id="filtro">
              <opcion valor="Todos" icono="✨" texto="Todos" contar="si"/>
              <opcion valor="Imágenes" icono="🖼️" texto="Imágenes" color="#8b5cf6" contar="si"/>
              <opcion valor="Música" icono="🎵" texto="Música" color="#ec4899" contar="si"/>
              <opcion valor="Videos" icono="🎬" texto="Videos" color="#f59e0b" contar="si"/>
              <opcion valor="Documentos" icono="📄" texto="Documentos" color="#3b82f6" contar="si"/>
            </botones>

            <horizontal clase="xd-controles">
              <desplegable id="campo" etiqueta="Ordenar por">
                <opcion valor="Nombre" texto="Nombre"/>
                <opcion valor="Tamaño" texto="Tamaño"/>
                <opcion valor="Fecha" texto="Fecha"/>
                <opcion valor="Extensión" texto="Extensión"/>
              </desplegable>
              <interruptor id="descendente" etiqueta="Descendente"/>
              <interruptor id="ver_todos" etiqueta="Mostrar todos"/>
            </horizontal>

            <html id="leyenda"/>
          </vertical>

          <html id="info"/>
          <html id="resultados"/>
        </vertical>

        <vertical id="vista_almacenamiento" clase="xd-vista xd-oculto">
          <html id="almacenamiento"/>
        </vertical>

      </vertical>

      <html id="splash" clase="xd-splash"><![CDATA[
<svg class="xd-olas" viewBox="0 0 2400 120" preserveAspectRatio="none"><g fill="rgba(255,255,255,.16)">
<path d="M0,60 C200,110 400,10 600,60 C800,110 1000,10 1200,60 L1200,120 L0,120Z"/>
<path transform="translate(1200)" d="M0,60 C200,110 400,10 600,60 C800,110 1000,10 1200,60 L1200,120 L0,120Z"/></g></svg>
<svg class="xd-olas b" viewBox="0 0 2400 120" preserveAspectRatio="none"><g fill="rgba(255,255,255,.12)">
<path d="M0,70 C250,20 350,110 600,70 C850,30 950,110 1200,70 L1200,120 L0,120Z"/>
<path transform="translate(1200)" d="M0,70 C250,20 350,110 600,70 C850,30 950,110 1200,70 L1200,120 L0,120Z"/></g></svg>
<div class="xd-splash-logo">📁</div>
<h2>Explorando tu Drive</h2>
<p>Preparando archivos, música y videos</p>
<div class="xd-splash-bar"><i></i></div>
      ]]></html>

      <html id="visor" clase="xd-visor"><![CDATA[
<div class="xd-visor-top">
  <div class="xd-visor-tit"><b id="visor-titulo"></b><span id="visor-meta"></span></div>
  <span class="xd-visor-cont" id="visor-cont"></span>
  <button class="xd-vb" id="visor-menos" title="Alejar">−</button>
  <button class="xd-vb" id="visor-mas" title="Acercar">+</button>
  <button class="xd-vb" id="visor-ajustar" title="Ajustar a la pantalla">⛶</button>
  <button class="xd-vb" id="visor-cerrar" title="Cerrar">✕</button>
</div>
<div class="xd-escena cargando" id="visor-escena">
  <button class="xd-nav xd-nav-i" id="visor-ant" title="Anterior">‹</button>
  <img id="visor-img" alt="" draggable="false">
  <div class="xd-spin"></div>
  <button class="xd-nav xd-nav-d" id="visor-sig" title="Siguiente">›</button>
</div>
      ]]></html>

      <html id="lector" clase="xd-lector"><![CDATA[
<div class="xd-lector-hoja">
  <div class="xd-lector-top">
    <span class="xd-ico" id="lector-ico">📄</span>
    <div class="xd-txt"><div class="xd-name" id="lector-titulo"></div><div class="xd-meta" id="lector-ext"></div></div>
    <button class="xd-vb" id="lector-menos" title="Letra más pequeña">A−</button>
    <button class="xd-vb" id="lector-mas" title="Letra más grande">A+</button>
    <button class="xd-vb" id="lector-cerrar" title="Cerrar">✕</button>
  </div>
  <div class="xd-lector-cuerpo" id="lector-cuerpo"></div>
</div>
      ]]></html>

      <html id="vwin" clase="xd-vwin"><![CDATA[<video id="xd-video" playsinline preload="metadata"></video>]]></html>

      <html id="mini" clase="xd-mini"><![CDATA[
<audio id="xd-audio" preload="metadata"></audio>
<div class="xd-mini-fila">
  <div class="xd-disco" id="mini-disco">🎵</div>
  <div class="xd-mini-info"><div class="xd-mini-titulo" id="mini-titulo"></div><div class="xd-mini-sub" id="mini-sub"></div></div>
  <div class="xd-eq"><i></i><i></i><i></i><i></i></div>
  <div class="xd-mini-ctl">
    <button class="xd-mb" id="mini-ant" title="Anterior">⏮</button>
    <button class="xd-mb xd-mb-play" id="mini-play" title="Reproducir o pausar">▶</button>
    <button class="xd-mb" id="mini-sig" title="Siguiente">⏭</button>
  </div>
  <label class="xd-vol" title="Volumen">🔊<input id="mini-vol" class="xd-rng" type="range" min="0" max="100" value="80"></label>
  <button class="xd-mb xd-solo-video" id="mini-grande" title="Ampliar o reducir el video">⤢</button>
  <button class="xd-mb" id="mini-cerrar" title="Cerrar reproductor">✕</button>
</div>
<div class="xd-mini-fila">
  <span id="mini-t0">0:00</span>
  <input id="mini-seek" class="xd-rng" type="range" min="0" max="1000" value="0">
  <span id="mini-t1">0:00</span>
</div>
      ]]></html>

    </vertical>
  </diseno>

  <script><![CDATA[
(function () {
  const $ = (id) => document.getElementById(id);
  const todos = (s, r) => Array.from((r || document).querySelectorAll(s));
  const python = async (nombre, args) =>
    (await google.colab.kernel.invokeFunction(nombre, args, {})).data['application/json'];
  const app = $('xd-app');
  const PDFJS = 'https://cdnjs.cloudflare.com/ajax/libs/pdf.js/3.11.174';
  let filtro = 'Todos', temporizador = null, ultima = 0, splashCerrado = false;

  /* ============ Servidor local (streaming con adelantar/retroceder) ============ */
  let base = null;
  const conectar = (async () => {
    try {
      if (!window.XD_PUERTO) return;
      const u = await google.colab.kernel.proxyPort(window.XD_PUERTO, { cache: true });
      base = String(u).replace(/\/$/, '');
    } catch (e) { base = null; }
  })();
  const listoServidor = Promise.race([conectar, new Promise((r) => setTimeout(r, 4000))]);
  const url = (tipo, id) => base ? base + '/' + tipo + '/' + window.XD_TOKEN + '/' + id : null;

  /* ============ Animación de entrada ============ */
  const t0 = performance.now();
  function cerrarSplash() {
    const resto = Math.max(0, 1900 - (performance.now() - t0));
    setTimeout(() => { $('splash').classList.add('fuera'); app.classList.add('listo'); }, resto);
  }
  setTimeout(() => { if (!splashCerrado) { splashCerrado = true; cerrarSplash(); } }, 8000);

  /* ============ Búsqueda y resultados ============ */
  async function buscar(animar) {
    const mia = ++ultima;
    try {
      const r = await python('explorador.buscar',
        [$('buscador').value, filtro, $('campo').value, $('descendente').checked, $('ver_todos').checked]);
      if (mia === ultima) {
        $('info').innerHTML = r.info;
        const res = $('resultados');
        res.innerHTML = r.html;
        if (animar) { res.classList.remove('xd-anim'); void res.offsetWidth; res.classList.add('xd-anim'); }
        refrescarFilas();
        await listoServidor;
        cargarMiniaturas(res);
      }
    } catch (e) {
      $('resultados').innerHTML = '<div class="xd-empty">⚠️ No se pudo consultar a Python.<br>' + e + '</div>';
    } finally {
      if (!splashCerrado) { splashCerrado = true; cerrarSplash(); }
    }
  }

  function cargarMiniaturas(raiz) {
    todos('img[data-mid]', raiz).forEach((img) => {
      const id = img.dataset.mid;
      img.loading = 'lazy';
      img.onload = () => { img.classList.add('ok'); img.parentNode.classList.add('listo'); };
      img.onerror = () => miniaturaRespaldo(img, id);
      const u = url('t', id);
      if (u) img.src = u; else miniaturaRespaldo(img, id);
    });
  }
  async function miniaturaRespaldo(img, id) {
    if (img.dataset.fb) { if (img.parentNode) img.parentNode.innerHTML = '<div class="xd-ph">🖼️</div>'; return; }
    img.dataset.fb = '1';
    try {
      const r = await python('explorador.miniatura', [Number(id)]);
      if (r.src) img.src = r.src; else img.onerror();
    } catch (e) { img.onerror(); }
  }

  $('buscador').addEventListener('input', () => { clearTimeout(temporizador); temporizador = setTimeout(() => buscar(false), 180); });
  ['campo', 'descendente', 'ver_todos'].forEach((id) => $(id).addEventListener('change', () => buscar(true)));
  $('filtro').addEventListener('click', (ev) => {
    const boton = ev.target.closest('.xd-btn');
    if (!boton) return;
    filtro = boton.dataset.valor;
    todos('.xd-btn', $('filtro')).forEach((b) => b.classList.toggle('activo', b === boton));
    buscar(true);
  });

  /* ============ Pestañas ============ */
  $('vistas').addEventListener('click', (ev) => {
    const b = ev.target.closest('.xd-tab');
    if (!b) return;
    todos('.xd-tab', $('vistas')).forEach((x) => x.classList.toggle('activo', x === b));
    const v = b.dataset.valor;
    $('vista_archivos').classList.toggle('xd-oculto', v !== 'archivos');
    $('vista_almacenamiento').classList.toggle('xd-oculto', v !== 'almacenamiento');
    if (v === 'almacenamiento') cargarAlmacenamiento();
  });

  /* ============ Mini reproductor (música y video) ============ */
  const audio = $('xd-audio'), video = $('xd-video'), mini = $('mini'), vwin = $('vwin');
  const seek = $('mini-seek'), vol = $('mini-vol');
  let cola = [], pos = -1, medio = null, actualId = null, intentoRespaldo = false;

  const fmt = (s) => {
    if (!isFinite(s)) return '0:00';
    const m = Math.floor(s / 60), x = Math.floor(s % 60);
    return m + ':' + (x < 10 ? '0' : '') + x;
  };
  const pintarRango = (el) => el.style.setProperty('--p', (el.value / el.max * 100) + '%');

  function subtitulo(t) {
    const it = cola[pos];
    $('mini-sub').textContent = t || (it ? (it.cat === 'Videos' ? 'Video' : 'Música') : '');
  }
  function refrescarFilas() {
    const sonando = medio && !medio.paused;
    todos('.xd-fila', $('resultados')).forEach((f) => {
      const b = f.querySelector('.xd-play:not(.xd-leer)');
      if (!b) return;
      const es = b.dataset.id === actualId;
      f.classList.toggle('actual', es);
      b.textContent = es && sonando ? '⏸' : '▶';
    });
  }
  function estadoPlay() {
    const sonando = !!medio && !medio.paused;
    $('mini-play').textContent = sonando ? '⏸' : '▶';
    mini.classList.toggle('sonando', sonando);
    refrescarFilas();
  }
  function actualizarTiempo() {
    if (!medio) return;
    const d = medio.duration || 0;
    seek.value = d ? medio.currentTime / d * 1000 : 0;
    pintarRango(seek);
    $('mini-t0').textContent = fmt(medio.currentTime);
    $('mini-t1').textContent = fmt(d);
  }

  async function reproducir(i) {
    pos = i;
    const it = cola[pos];
    if (!it) return;
    audio.pause(); video.pause();
    intentoRespaldo = false;
    actualId = it.id;
    medio = it.cat === 'Videos' ? video : audio;
    mini.classList.add('activo');
    mini.classList.toggle('es-video', it.cat === 'Videos');
    vwin.classList.toggle('activo', it.cat === 'Videos');
    $('mini-titulo').textContent = it.nombre;
    $('mini-disco').textContent = it.cat === 'Videos' ? '🎬' : '🎵';
    subtitulo();
    seek.value = 0; pintarRango(seek);
    $('mini-t0').textContent = '0:00'; $('mini-t1').textContent = '0:00';
    await listoServidor;
    const u = url('f', it.id);
    if (u) { medio.src = u; medio.play().catch(() => {}); } else { await respaldo(it); }
    refrescarFilas();
  }
  async function respaldo(it) {
    subtitulo('Cargando archivo...');
    try {
      const r = await python('explorador.reproducir', [Number(it.id)]);
      if (r.error) { subtitulo(r.error); return; }
      subtitulo('');
      medio.src = r.src;
      medio.play().catch(() => {});
    } catch (e) { subtitulo('No se pudo cargar el archivo.'); }
  }
  function togglePlay() {
    if (!medio) return;
    if (medio.paused) medio.play().catch(() => {}); else medio.pause();
  }
  function siguiente(auto) {
    if (pos < cola.length - 1) reproducir(pos + 1);
    else if (!auto && cola.length) reproducir(0);
  }
  function anterior() {
    if (medio && medio.currentTime > 3) { medio.currentTime = 0; return; }
    if (pos > 0) reproducir(pos - 1);
  }
  function cerrarReproductor() {
    audio.pause(); video.pause();
    audio.removeAttribute('src'); video.removeAttribute('src');
    audio.load(); video.load();
    mini.classList.remove('activo', 'sonando');
    vwin.classList.remove('activo', 'grande');
    medio = null; actualId = null; pos = -1;
    refrescarFilas();
  }

  [audio, video].forEach((m) => {
    m.addEventListener('timeupdate', () => { if (m === medio) actualizarTiempo(); });
    m.addEventListener('loadedmetadata', () => { if (m === medio) actualizarTiempo(); });
    m.addEventListener('play', () => { if (m === medio) estadoPlay(); });
    m.addEventListener('pause', () => { if (m === medio) estadoPlay(); });
    m.addEventListener('ended', () => { if (m === medio) siguiente(true); });
    m.addEventListener('error', async () => {
      if (m !== medio || !m.getAttribute('src')) return;
      const it = cola[pos];
      if (!it) return;
      if (!intentoRespaldo) { intentoRespaldo = true; await respaldo(it); }
      else subtitulo('Tu navegador no puede reproducir este formato.');
    });
    m.volume = 0.8;
  });
  video.addEventListener('click', togglePlay);
  $('mini-play').addEventListener('click', togglePlay);
  $('mini-sig').addEventListener('click', () => siguiente(false));
  $('mini-ant').addEventListener('click', anterior);
  $('mini-cerrar').addEventListener('click', cerrarReproductor);
  $('mini-grande').addEventListener('click', () => vwin.classList.toggle('grande'));
  seek.addEventListener('input', () => {
    pintarRango(seek);
    if (medio && medio.duration) medio.currentTime = seek.value / 1000 * medio.duration;
  });
  vol.addEventListener('input', () => { pintarRango(vol); audio.volume = video.volume = vol.value / 100; });
  pintarRango(vol);

  /* ============ Visor de imágenes ============ */
  const visor = $('visor'), vimg = $('visor-img'), escena = $('visor-escena');
  let galeria = [], gi = 0, zoom = 1, px = 0, py = 0, arrastre = null;

  function aplicarZoom() { vimg.style.transform = 'translate(' + px + 'px,' + py + 'px) scale(' + zoom + ')'; }
  function ajustar() { zoom = 1; px = 0; py = 0; aplicarZoom(); }
  function cambiarZoom(f) {
    zoom = Math.min(8, Math.max(1, zoom * f));
    if (zoom === 1) { px = 0; py = 0; }
    aplicarZoom();
  }
  function abrirVisor(tarjeta) {
    galeria = todos('.xd-card[data-id]', $('resultados')).map((c) => ({
      id: c.dataset.id,
      nombre: c.querySelector('.xd-cap b').textContent,
      meta: c.querySelector('.xd-cap span').textContent }));
    gi = Math.max(0, galeria.findIndex((g) => g.id === tarjeta.dataset.id));
    visor.classList.add('abierto');
    mostrarImagen();
  }
  function mostrarImagen() {
    const g = galeria[gi];
    ajustar();
    $('visor-titulo').textContent = g.nombre;
    $('visor-meta').textContent = g.meta;
    $('visor-cont').textContent = (gi + 1) + ' / ' + galeria.length;
    escena.classList.add('cargando');
    vimg.classList.remove('ok');
    vimg.dataset.fb = '';
    vimg.onload = () => { escena.classList.remove('cargando'); vimg.classList.add('ok'); };
    vimg.onerror = async () => {
      if (vimg.dataset.fb) { escena.classList.remove('cargando'); return; }
      vimg.dataset.fb = '1';
      try {
        const r = await python('explorador.imagen_grande', [Number(g.id)]);
        if (r.src) vimg.src = r.src; else escena.classList.remove('cargando');
      } catch (e) { escena.classList.remove('cargando'); }
    };
    const u = url('f', g.id);
    if (u) vimg.src = u; else vimg.onerror();
    const vecina = galeria[(gi + 1) % galeria.length];
    const uv = url('f', vecina.id);
    if (uv) { const pre = new Image(); pre.src = uv; }
  }
  function moverVisor(d) { gi = (gi + d + galeria.length) % galeria.length; mostrarImagen(); }
  function cerrarVisor() { visor.classList.remove('abierto'); }

  $('visor-cerrar').addEventListener('click', cerrarVisor);
  $('visor-ant').addEventListener('click', () => moverVisor(-1));
  $('visor-sig').addEventListener('click', () => moverVisor(1));
  $('visor-mas').addEventListener('click', () => cambiarZoom(1.35));
  $('visor-menos').addEventListener('click', () => cambiarZoom(1 / 1.35));
  $('visor-ajustar').addEventListener('click', ajustar);
  escena.addEventListener('wheel', (ev) => { ev.preventDefault(); cambiarZoom(ev.deltaY < 0 ? 1.15 : 1 / 1.15); }, { passive: false });
  vimg.addEventListener('dblclick', () => { if (zoom > 1) ajustar(); else cambiarZoom(2.5); });
  vimg.addEventListener('pointerdown', (ev) => {
    if (zoom <= 1) return;
    arrastre = { x: ev.clientX - px, y: ev.clientY - py };
    vimg.setPointerCapture(ev.pointerId);
    vimg.classList.add('arrastrando');
  });
  vimg.addEventListener('pointermove', (ev) => {
    if (!arrastre) return;
    px = ev.clientX - arrastre.x; py = ev.clientY - arrastre.y; aplicarZoom();
  });
  const soltar = () => { arrastre = null; vimg.classList.remove('arrastrando'); };
  vimg.addEventListener('pointerup', soltar);
  vimg.addEventListener('pointercancel', soltar);

  /* ============ Lector de documentos ============ */
  const lector = $('lector'), cuerpo = $('lector-cuerpo');
  let pdfLib = null, tokenDoc = 0, tamLetra = 16;

  async function cargarPdfJs() {
    if (pdfLib) return pdfLib;
    await new Promise((ok, mal) => {
      const s = document.createElement('script');
      s.src = PDFJS + '/pdf.min.js';
      s.onload = ok;
      s.onerror = () => mal(new Error('No se pudo cargar PDF.js (¿sin conexión?)'));
      document.head.appendChild(s);
    });
    const lib = window.pdfjsLib;
    try {
      const txt = await (await fetch(PDFJS + '/pdf.worker.min.js')).text();
      lib.GlobalWorkerOptions.workerSrc = URL.createObjectURL(new Blob([txt], { type: 'text/javascript' }));
    } catch (e) { lib.GlobalWorkerOptions.workerSrc = PDFJS + '/pdf.worker.min.js'; }
    pdfLib = lib;
    return lib;
  }
  async function mostrarPdf(b64, mio) {
    const lib = await cargarPdfJs();
    const bin = atob(b64);
    const datos = new Uint8Array(bin.length);
    for (let i = 0; i < bin.length; i++) datos[i] = bin.charCodeAt(i);
    const doc = await lib.getDocument({ data: datos }).promise;
    cuerpo.innerHTML = '';
    cuerpo.classList.add('pdf');
    const ancho = Math.max(200, Math.min(cuerpo.clientWidth - 36, 900));
    const dpr = window.devicePixelRatio || 1;
    const tope = Math.min(doc.numPages, 60);
    for (let n = 1; n <= tope; n++) {
      if (mio !== tokenDoc) return;
      const pag = await doc.getPage(n);
      const escala = ancho / pag.getViewport({ scale: 1 }).width;
      const v = pag.getViewport({ scale: escala * dpr });
      const c = document.createElement('canvas');
      c.className = 'xd-pagina';
      c.width = v.width; c.height = v.height;
      c.style.width = ancho + 'px'; c.style.height = (v.height / dpr) + 'px';
      cuerpo.appendChild(c);
      await pag.render({ canvasContext: c.getContext('2d'), viewport: v }).promise;
    }
    if (doc.numPages > tope) {
      cuerpo.insertAdjacentHTML('beforeend', '<div class="xd-nota">Se muestran las primeras ' + tope + ' de ' + doc.numPages + ' páginas.</div>');
    }
  }
  async function abrirDoc(id) {
    const mio = ++tokenDoc;
    lector.classList.add('abierto');
    cuerpo.className = 'xd-lector-cuerpo';
    cuerpo.style.setProperty('--fs', tamLetra + 'px');
    cuerpo.scrollTop = 0;
    cuerpo.innerHTML = '<div class="xd-cargando">⏳ Abriendo documento...</div>';
    try {
      const r = await python('explorador.leer', [Number(id)]);
      if (mio !== tokenDoc) return;
      $('lector-titulo').textContent = r.titulo;
      $('lector-ext').textContent = r.ext;
      $('lector-ico').textContent = r.icono;
      if (r.tipo === 'pdf') await mostrarPdf(r.datos, mio); else cuerpo.innerHTML = r.html;
    } catch (e) {
      if (mio === tokenDoc) cuerpo.innerHTML = '<div class="xd-empty">⚠️ No se pudo abrir el documento.<br>' + e + '</div>';
    }
  }
  function cerrarLector() { tokenDoc++; lector.classList.remove('abierto'); }
  $('lector-cerrar').addEventListener('click', cerrarLector);
  $('lector-mas').addEventListener('click', () => { tamLetra = Math.min(26, tamLetra + 2); cuerpo.style.setProperty('--fs', tamLetra + 'px'); });
  $('lector-menos').addEventListener('click', () => { tamLetra = Math.max(11, tamLetra - 2); cuerpo.style.setProperty('--fs', tamLetra + 'px'); });

  /* ============ Clics en los resultados ============ */
  $('resultados').addEventListener('click', (ev) => {
    const boton = ev.target.closest('.xd-play');
    if (boton) {
      if (boton.classList.contains('xd-leer')) { abrirDoc(boton.dataset.id); return; }
      if (boton.dataset.id === actualId && medio) { togglePlay(); return; }
      const cat = boton.dataset.cat;
      cola = todos('.xd-play[data-cat="' + cat + '"]', $('resultados')).map((b) => ({
        id: b.dataset.id, cat: cat,
        nombre: b.closest('.xd-fila').querySelector('.xd-name').textContent }));
      reproducir(Math.max(0, cola.findIndex((x) => x.id === boton.dataset.id)));
      return;
    }
    const tarjeta = ev.target.closest('.xd-card[data-id]');
    if (tarjeta) abrirVisor(tarjeta);
  });

  /* ============ Teclado ============ */
  document.addEventListener('keydown', (ev) => {
    const t = ev.target.tagName;
    if (t === 'INPUT' && ev.target.type === 'text') return;
    if (visor.classList.contains('abierto')) {
      if (ev.key === 'Escape') cerrarVisor();
      else if (ev.key === 'ArrowLeft') moverVisor(-1);
      else if (ev.key === 'ArrowRight') moverVisor(1);
      else if (ev.key === '+' || ev.key === '=') cambiarZoom(1.35);
      else if (ev.key === '-') cambiarZoom(1 / 1.35);
      else if (ev.key === '0') ajustar();
    } else if (lector.classList.contains('abierto')) {
      if (ev.key === 'Escape') cerrarLector();
    } else if (ev.code === 'Space' && medio && t !== 'INPUT' && t !== 'SELECT' && t !== 'BUTTON') {
      ev.preventDefault();
      togglePlay();
    }
  });

  /* ============ Almacenamiento ============ */
  async function cargarAlmacenamiento() {
    const cont = $('almacenamiento');
    if (!cont.innerHTML) cont.innerHTML = '<div class="xd-cargando">⏳ Calculando almacenamiento...</div>';
    try {
      const d = await python('explorador.almacenamiento', []);
      cont.innerHTML = htmlAlmacenamiento(d);
      requestAnimationFrame(() => requestAnimationFrame(() => animarAlmacenamiento(cont)));
    } catch (e) {
      cont.innerHTML = '<div class="xd-empty">⚠️ No se pudo calcular el almacenamiento.<br>' + e + '</div>';
    }
  }
  function htmlAlmacenamiento(d) {
    const R = 80, C = 2 * Math.PI * R;
    let acum = 0, segs = '', ley = '';
    d.categorias.forEach((c, i) => {
      const largo = C * c.pct / 100;
      const visible = Math.max(0, largo - (c.pct > 0 && c.pct < 100 ? 2 : 0));
      segs += '<circle class="xd-seg" cx="100" cy="100" r="' + R + '" fill="none" stroke="' + c.color +
        '" stroke-width="26" data-len="' + visible + '" data-c="' + C + '" stroke-dasharray="0 ' + C +
        '" stroke-dashoffset="' + (-acum) + '" style="transition-delay:' + (i * 140) + 'ms"/>';
      acum += largo;
      ley += '<div class="xd-ley"><i style="background:' + c.color + '"></i>' + c.icono + ' ' + c.nombre +
        '<span>' + c.fmt + ' · ' + c.n + ' arch. · ' + c.pct.toFixed(1) + '%</span></div>';
    });
    let top = '';
    d.top.forEach((a) => {
      top += '<div class="xd-top"><span class="n" title="' + a.nombre + '">' + a.icono + ' ' + a.nombre + '</span>' +
        '<div class="xd-barra"><i data-w="' + a.pct + '" style="background:' + a.color + '"></i></div><span class="t">' + a.fmt + '</span></div>';
    });
    let drive = '';
    if (d.drive) {
      drive = '<div class="xd-caja xd-ancha"><h3>☁️ Tu Google Drive</h3>' +
        '<div class="xd-barra xd-drive-barra"><i data-w="' + d.drive.pct + '"></i></div>' +
        '<div class="xd-meta">' + d.drive.usado + ' usados de ' + d.drive.total + ' · ' + d.drive.libre + ' libres (' + d.drive.pct.toFixed(1) + '%)</div></div>';
    }
    return '<div class="xd-alm">' +
      '<div class="xd-caja"><h3>Espacio por tipo de archivo</h3><div class="xd-dona">' +
        '<div class="xd-dona-svg"><svg viewBox="0 0 200 200"><circle cx="100" cy="100" r="' + R +
        '" fill="none" stroke="rgba(128,128,128,.2)" stroke-width="26"/>' + segs + '</svg>' +
        '<div class="xd-dona-c"><b>' + d.total_fmt + '</b><span>' + d.n + ' archivos</span></div></div>' +
        '<div class="xd-leyenda">' + ley + '</div></div></div>' +
      '<div class="xd-caja"><h3>Los archivos más pesados</h3>' + top +
        '<div class="xd-meta" style="margin-top:12px">Ordenados con Merge Sort en ' + d.ms + ' ms</div></div>' +
      drive + '</div>';
  }
  function animarAlmacenamiento(cont) {
    todos('.xd-seg', cont).forEach((s) => s.setAttribute('stroke-dasharray', s.dataset.len + ' ' + s.dataset.c));
    todos('.xd-barra i[data-w]', cont).forEach((b) => { b.style.width = b.dataset.w + '%'; });
  }

  buscar(false);
})();
  ]]></script>

</interfaz>
'''

PLANTILLAS = {}
COLOR = {"Imágenes": "#8b5cf6", "Música": "#ec4899", "Videos": "#f59e0b", "Documentos": "#3b82f6"}
IDS_REQUERIDOS = ["buscador", "filtro", "campo", "descendente", "ver_todos", "info", "resultados",
                  "vistas", "vista_archivos", "vista_almacenamiento", "almacenamiento",
                  "splash", "visor", "lector", "vwin", "mini"]
archivos = []                         # se llena en la ejecución principal


def cargar_xml():
    """Lee el diseño con xml.etree.ElementTree (incluido en Python)."""
    if os.path.exists(XML_ARCHIVO):
        print(f"📄 Usando el diseño del archivo '{XML_ARCHIVO}'")
        return ET.parse(XML_ARCHIVO).getroot()
    return ET.fromstring(INTERFAZ_XML.encode("utf-8"))


def render(plantilla, /, **valores):
    return Template(PLANTILLAS[plantilla]).safe_substitute(valores)


# ---------- Convierte el árbol <diseno> del XML en HTML ----------
def _estilo(nodo):
    return f' style="width:{nodo.get("ancho")}"' if nodo.get("ancho") else ""


def xml_a_html(nodo, datos):
    esc = lambda t: html.escape(t or "", quote=True)
    etiqueta, ident = nodo.tag, nodo.get("id")
    id_attr = f' id="{esc(ident)}"' if ident else ""
    clases = esc(nodo.get("clase", ""))

    if etiqueta in ("vertical", "horizontal"):
        hijos = "".join(xml_a_html(h, datos) for h in nodo)
        return f'<div{id_attr} class="{clases}"{_estilo(nodo)}>{hijos}</div>'

    if etiqueta == "html":
        contenido = Template((nodo.text or "").strip()).safe_substitute(datos)
        if not contenido and ident in datos["_bloques"]:
            contenido = datos["_bloques"][ident]
        return f'<div{id_attr} class="{clases}">{contenido}</div>'

    if etiqueta == "texto":
        icono = f'<span class="xd-input-i">{esc(nodo.get("icono"))}</span>' if nodo.get("icono") else ""
        return (f'<div class="xd-input">{icono}<input{id_attr} type="text" autocomplete="off" '
                f'placeholder="{esc(nodo.get("ayuda"))}"></div>')

    if etiqueta == "botones":
        botones = []
        for i, o in enumerate(nodo.findall("opcion")):
            valor = o.get("valor")
            cuenta = ""
            if o.get("contar") == "si":
                n = datos["_total"] if valor == "Todos" else datos["_conteo"].get(valor, 0)
                cuenta = f'<span class="xd-n">{n}</span>'
            color = f' style="--c:{esc(o.get("color"))}"' if o.get("color") else ""
            activo = " activo" if i == 0 else ""
            botones.append(f'<button class="xd-btn{activo}" data-valor="{esc(valor)}"{color}>'
                           f'{esc(o.get("icono"))} {esc(o.get("texto"))}{cuenta}</button>')
        return f'<div{id_attr} class="xd-botones">{"".join(botones)}</div>'

    if etiqueta == "pestanas":
        botones = []
        for i, o in enumerate(nodo.findall("opcion")):
            activo = " activo" if i == 0 else ""
            botones.append(f'<button class="xd-tab{activo}" data-valor="{esc(o.get("valor"))}">'
                           f'{esc(o.get("icono"))} {esc(o.get("texto"))}</button>')
        return f'<div{id_attr} class="xd-tabs">{"".join(botones)}</div>'

    if etiqueta == "desplegable":
        opciones = "".join(f'<option value="{esc(o.get("valor"))}">{esc(o.get("texto"))}</option>'
                           for o in nodo.findall("opcion"))
        return (f'<label class="xd-select"><span>{esc(nodo.get("etiqueta"))}</span>'
                f'<select{id_attr}>{opciones}</select></label>')

    if etiqueta == "interruptor":
        return (f'<label class="xd-switch"><input{id_attr} type="checkbox"><span class="xd-slider"></span>'
                f'<span>{esc(nodo.get("etiqueta"))}</span></label>')

    raise ValueError(f"Elemento desconocido en el XML: <{etiqueta}>")


# ============================================================
# 5. SERVIDOR LOCAL DE ARCHIVOS (streaming fluido, sin base64)
# ============================================================
# El navegador pide /f/<token>/<id> (archivo original, con soporte de Range para adelantar
# y retroceder en audio/video) o /t/<token>/<id> (miniatura JPEG). Colab lo expone con
# google.colab.kernel.proxyPort(). Solo sirve los archivos escaneados, identificados por id.
TOKEN = secrets.token_urlsafe(8)
PUERTO = 0
_servidor = None
_CACHE_MINIATURAS = {}


def miniatura_bytes(ruta):
    if ruta not in _CACHE_MINIATURAS:
        try:
            with PILImage.open(ruta) as im:
                im = ImageOps.exif_transpose(im).convert("RGB")
                im.thumbnail((360, 360))
                buffer = io.BytesIO()
                im.save(buffer, "JPEG", quality=80)
            _CACHE_MINIATURAS[ruta] = buffer.getvalue()
        except Exception:
            _CACHE_MINIATURAS[ruta] = None
    return _CACHE_MINIATURAS[ruta]


class ManejadorArchivos(BaseHTTPRequestHandler):
    protocol_version = "HTTP/1.1"

    def log_message(self, *args):
        pass

    def _buscar(self):
        partes = self.path.split('?')[0].strip('/').split('/')
        if len(partes) == 3 and partes[0] in ('f', 't') and partes[1] == TOKEN and partes[2].isdigit():
            i = int(partes[2])
            if 0 <= i < len(archivos):
                return partes[0], archivos[i]
        return None, None

    def _enviar_bytes(self, datos, mime):
        self.send_response(200)
        self.send_header("Content-Type", mime)
        self.send_header("Content-Length", str(len(datos)))
        self.send_header("Cache-Control", "private, max-age=3600")
        self.send_header("Access-Control-Allow-Origin", "*")
        self.end_headers()
        self.wfile.write(datos)

    def do_GET(self):
        try:
            tipo, a = self._buscar()
            if not a:
                self.send_error(404)
                return
            if tipo == 't':
                datos = miniatura_bytes(a.ruta)
                if datos is None:
                    self.send_error(404)
                else:
                    self._enviar_bytes(datos, "image/jpeg")
                return

            tam = os.path.getsize(a.ruta)
            mime = mimetypes.guess_type(a.ruta)[0] or "application/octet-stream"
            ini, fin, estado = 0, max(tam - 1, 0), 200
            rango = self.headers.get("Range")
            if rango and tam > 0:
                m = re.match(r"bytes=(\d*)-(\d*)", rango)
                if m:
                    g1, g2 = m.groups()
                    if g1 == "" and g2 != "":
                        ini = max(0, tam - int(g2))
                    elif g1 != "":
                        ini = int(g1)
                        if g2 != "":
                            fin = min(int(g2), tam - 1)
                    if ini > fin or ini >= tam:
                        self.send_response(416)
                        self.send_header("Content-Range", f"bytes */{tam}")
                        self.send_header("Content-Length", "0")
                        self.end_headers()
                        return
                    estado = 206
            largo = 0 if tam == 0 else fin - ini + 1
            self.send_response(estado)
            self.send_header("Content-Type", mime)
            self.send_header("Accept-Ranges", "bytes")
            self.send_header("Content-Length", str(largo))
            self.send_header("Cache-Control", "private, max-age=3600")
            self.send_header("Access-Control-Allow-Origin", "*")
            if estado == 206:
                self.send_header("Content-Range", f"bytes {ini}-{fin}/{tam}")
            self.end_headers()
            if largo:
                with open(a.ruta, "rb") as f:
                    f.seek(ini)
                    restante = largo
                    while restante > 0:
                        bloque = f.read(min(262144, restante))
                        if not bloque:
                            break
                        self.wfile.write(bloque)
                        restante -= len(bloque)
        except (BrokenPipeError, ConnectionResetError):
            pass            # el navegador cortó la descarga (normal al adelantar un video)
        except Exception:
            try:
                self.send_error(500)
            except Exception:
                pass


def iniciar_servidor():
    global _servidor, PUERTO
    try:
        _servidor = ThreadingHTTPServer(("127.0.0.1", 0), ManejadorArchivos)
        _servidor.daemon_threads = True
        PUERTO = _servidor.server_address[1]
        threading.Thread(target=_servidor.serve_forever, daemon=True).start()
    except Exception as e:
        PUERTO = 0
        print(f"⚠️ No se pudo iniciar el servidor local ({e}). Se usará el modo de respaldo (más lento).")


# ============================================================
# 6. BACKEND EN PYTHON (callbacks que llama la interfaz)
# ============================================================
def data_uri_imagen(ruta, maximo):
    try:
        with PILImage.open(ruta) as im:
            im = ImageOps.exif_transpose(im).convert("RGB")
            im.thumbnail((maximo, maximo))
            buffer = io.BytesIO()
            im.save(buffer, "JPEG", quality=85)
        return "data:image/jpeg;base64," + base64.b64encode(buffer.getvalue()).decode()
    except Exception:
        return None


def html_imagenes(grupo):
    tarjetas = [render("tarjeta-imagen", id=a.id, nombre=html.escape(a.nombre),
                       tamano=formatear_tamano(a.tamano), fecha=a.fecha[:10]) for a in grupo]
    return render("cuadricula", tarjetas="".join(tarjetas))


def html_fila(a):
    if a.categoria in ("Música", "Videos"):
        boton = render("boton-play", id=a.id, cat=a.categoria)
    elif a.categoria == "Documentos" and a.extension in LEGIBLES:
        boton = render("boton-leer", id=a.id)
    else:
        boton = ""
    return render("fila-archivo", color=COLOR[a.categoria], icono=a.icono, nombre=html.escape(a.nombre),
                  tamano=formatear_tamano(a.tamano), fecha=a.fecha, ext=a.extension.strip(".") or "archivo",
                  boton=boton)


def buscar(texto, filtro, campo, descendente, ver_todos):
    """Filtra y ordena (cada tipo con SU algoritmo) y devuelve el HTML ya armado."""
    texto = (texto or "").strip().lower()
    lista = [a for a in archivos if texto in a.nombre.lower() and filtro in ("Todos", a.categoria)]
    por_categoria = ordenar_por_categoria(lista, campo, bool(descendente))

    partes = []
    for categoria in CATEGORIAS:
        if categoria not in por_categoria:
            continue
        grupo, algoritmo, ms, comps = por_categoria[categoria]
        visibles = grupo if ver_todos else grupo[:MAX_POR_CATEGORIA]
        partes.append(render("seccion", color=COLOR[categoria], icono=ICONO[categoria], categoria=categoria,
                             total=len(grupo), motivo=html.escape(MOTIVO[categoria]), algoritmo=algoritmo,
                             ms=f"{ms:.3f}", comparaciones=comps))
        if categoria == "Imágenes":
            partes.append(html_imagenes(visibles))
        else:
            partes.append("".join(html_fila(a) for a in visibles))
        if len(grupo) > len(visibles):
            partes.append(render("mas", cantidad=len(grupo) - len(visibles)))

    total_ms = sum(r[2] for r in por_categoria.values())
    return JSON({"info": render("info", ms=f"{total_ms:.3f}", total=len(lista)),
                 "html": "".join(partes) if lista else render("vacio")})


# ----- Respaldos (solo se usan si el servidor local no está disponible) -----
def miniatura(id_archivo):
    return JSON({"src": data_uri_imagen(archivos[int(id_archivo)].ruta, 360)})


def imagen_grande(id_archivo):
    return JSON({"src": data_uri_imagen(archivos[int(id_archivo)].ruta, 1800)})


def reproducir(id_archivo):
    a = archivos[int(id_archivo)]
    es_musica = a.categoria == "Música"
    if a.tamano > (LIMITE_MUSICA if es_musica else LIMITE_VIDEO):
        return JSON({"error": f"Pesa {formatear_tamano(a.tamano)}: ábrelo desde Drive."})
    mime = mimetypes.guess_type(a.ruta)[0] or ("audio/mpeg" if es_musica else "video/mp4")
    with open(a.ruta, "rb") as f:
        return JSON({"src": f"data:{mime};base64," + base64.b64encode(f.read()).decode()})


# ----- Lector de documentos -----
def _aviso(mensaje):
    return render("aviso", mensaje=html.escape(mensaje))


def _leer_texto(ruta):
    with open(ruta, "r", encoding="utf-8", errors="replace") as f:
        texto = f.read(LIMITE_TEXTO + 1)
    cortado = len(texto) > LIMITE_TEXTO
    return texto[:LIMITE_TEXTO], cortado


def _nota_corte(cortado):
    return '<div class="xd-nota" style="margin-top:14px">Vista parcial: el archivo es más largo.</div>' if cortado else ""


def tabla_csv(ruta, delimitador):
    filas, cortado = [], False
    with open(ruta, "r", encoding="utf-8", errors="replace", newline="") as f:
        for i, fila in enumerate(csv.reader(f, delimiter=delimitador)):
            if i >= 300:
                cortado = True
                break
            filas.append(fila)
    if not filas:
        return _aviso("El archivo está vacío.")
    cabecera = "".join(f"<th>{html.escape(c)}</th>" for c in filas[0])
    cuerpo = "".join("<tr>" + "".join(f"<td>{html.escape(c)}</td>" for c in fila) + "</tr>" for fila in filas[1:])
    return (f'<div style="overflow:auto"><table><thead><tr>{cabecera}</tr></thead><tbody>{cuerpo}</tbody></table></div>'
            + _nota_corte(cortado))


def docx_a_html(ruta):
    """Extrae el texto de un .docx sin librerías externas (un .docx es un zip con XML)."""
    ns = "{http://schemas.openxmlformats.org/wordprocessingml/2006/main}"
    with zipfile.ZipFile(ruta) as z:
        raiz = ET.fromstring(z.read("word/document.xml"))
    partes = []
    for p in raiz.iter(ns + "p"):
        texto = "".join(t.text or "" for t in p.iter(ns + "t"))
        if not texto.strip():
            continue
        estilo = p.find(f"{ns}pPr/{ns}pStyle")
        nombre = (estilo.get(ns + "val") if estilo is not None else "") or ""
        etiqueta = "h3" if nombre.lower().startswith(("heading", "title", "ttulo", "t\u00edtulo")) else "p"
        partes.append(f"<{etiqueta}>{html.escape(texto)}</{etiqueta}>")
    return "".join(partes) or _aviso("No se encontró texto en el documento.")


def leer(id_archivo):
    a = archivos[int(id_archivo)]
    base = {"titulo": a.nombre, "icono": a.icono,
            "ext": f"{a.extension.strip('.').upper()} · {formatear_tamano(a.tamano)}"}
    try:
        ext = a.extension
        if ext == ".pdf":
            if a.tamano > LIMITE_PDF:
                return JSON({**base, "tipo": "html", "html": _aviso(f"El PDF pesa {formatear_tamano(a.tamano)}; ábrelo desde Drive.")})
            with open(a.ruta, "rb") as f:
                return JSON({**base, "tipo": "pdf", "datos": base64.b64encode(f.read()).decode()})
        if ext == ".docx":
            contenido = docx_a_html(a.ruta)
        elif ext in (".csv", ".tsv"):
            contenido = tabla_csv(a.ruta, "\t" if ext == ".tsv" else ",")
        elif ext == ".json":
            texto, cortado = _leer_texto(a.ruta)
            try:
                texto = json.dumps(json.loads(texto), indent=2, ensure_ascii=False)
            except ValueError:
                pass
            contenido = f"<pre>{html.escape(texto)}</pre>" + _nota_corte(cortado)
        else:
            texto, cortado = _leer_texto(a.ruta)
            contenido = f"<pre>{html.escape(texto)}</pre>" + _nota_corte(cortado)
    except Exception as e:
        contenido = _aviso(f"No se pudo abrir el documento: {e}")
    return JSON({**base, "tipo": "html", "html": contenido})


# ----- Gráfico de almacenamiento -----
def almacenamiento():
    total = sum(a.tamano for a in archivos)
    categorias = []
    for c in CATEGORIAS:
        grupo = [a for a in archivos if a.categoria == c]
        peso = sum(a.tamano for a in grupo)
        categorias.append({"nombre": c, "icono": ICONO[c], "color": COLOR[c], "n": len(grupo),
                           "fmt": formatear_tamano(peso), "pct": round(peso * 100 / total, 2) if total else 0})

    mayores, ms, _ = ordenar(archivos, "Merge Sort", "Tamaño", True)
    mayores = mayores[:6]
    maximo = mayores[0].tamano if mayores else 0
    top = [{"nombre": html.escape(a.nombre), "icono": a.icono, "color": COLOR[a.categoria],
            "fmt": formatear_tamano(a.tamano), "pct": round(a.tamano * 100 / maximo, 2) if maximo else 0}
           for a in mayores]

    drive_info = None
    try:
        u = shutil.disk_usage("/content/drive/MyDrive")
        if u.total > 0:
            drive_info = {"usado": formatear_tamano(u.used), "total": formatear_tamano(u.total),
                          "libre": formatear_tamano(u.free), "pct": round(u.used * 100 / u.total, 2)}
    except Exception:
        pass

    return JSON({"total_fmt": formatear_tamano(total), "n": len(archivos), "categorias": categorias,
                 "top": top, "ms": f"{ms:.3f}", "drive": drive_info})


# ============================================================
# 7. INTERFAZ Y EJECUCIÓN PRINCIPAL
# ============================================================
def construir_html():
    raiz = cargar_xml()
    PLANTILLAS.clear()
    PLANTILLAS.update({p.get("id"): (p.text or "").strip() for p in raiz.findall("plantillas/plantilla")})

    conteo = {c: sum(a.categoria == c for a in archivos) for c in CATEGORIAS}
    estadisticas = "".join(render("estadistica", icono=ICONO[c], n=conteo[c], nombre=c.lower()) for c in CATEGORIAS)
    estadisticas += render("estadistica", icono="💾", n=formatear_tamano(sum(a.tamano for a in archivos)), nombre="en total")
    leyenda = render("leyenda", items="".join(render(
        "leyenda-item", color=COLOR[c], motivo=html.escape(MOTIVO[c]), icono=ICONO[c],
        categoria=c, algoritmo=ALGORITMO_POR_CATEGORIA[c]) for c in CATEGORIAS))
    datos = {"carpeta": html.escape(NOMBRE_CARPETA), "estadisticas": estadisticas,
             "_conteo": conteo, "_total": len(archivos), "_bloques": {"leyenda": leyenda}}

    pantalla = xml_a_html(raiz.find("diseno")[0], datos)
    ids = {e.get("id") for e in raiz.find("diseno").iter() if e.get("id")}
    faltan = [i for i in IDS_REQUERIDOS if i not in ids]
    if faltan:
        raise ValueError(f"Al XML le faltan estos elementos con id: {faltan}")

    return (f"<style>{raiz.findtext('estilos')}</style>"
            f'<div id="xd-app">{pantalla}</div>'
            f'<script>window.XD_PUERTO={PUERTO};window.XD_TOKEN="{TOKEN}";</script>'
            f"<script>{raiz.findtext('script')}</script>")


def main():
    print("🔐 Conectando con Google Drive...")
    drive.mount('/content/drive')

    if not os.path.isdir(DIRECTORIO):
        print(f"❌ No se encontró la carpeta '{NOMBRE_CARPETA}' en tu Drive.")
        raiz = '/content/drive/MyDrive'
        carpetas = sorted(n for n in os.listdir(raiz) if os.path.isdir(os.path.join(raiz, n)))
        print("📂 Carpetas disponibles en MyDrive:", ", ".join(carpetas) or "(ninguna)")
        return

    archivos[:] = escanear(DIRECTORIO)
    print(f"✅ {len(archivos)} archivos encontrados en '{NOMBRE_CARPETA}'.")
    if archivos:
        iniciar_servidor()
        output.register_callback("explorador.buscar", buscar)
        output.register_callback("explorador.miniatura", miniatura)
        output.register_callback("explorador.imagen_grande", imagen_grande)
        output.register_callback("explorador.reproducir", reproducir)
        output.register_callback("explorador.leer", leer)
        output.register_callback("explorador.almacenamiento", almacenamiento)
        display(HTML(construir_html()))


if __name__ == "__main__":
    main()

🔐 Conectando con Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ 32 archivos encontrados en 'Acoplamiento'.
